# Install and import libraries

In [1]:
import sys
print(sys.executable)

/Users/ioana/.pyenv/versions/tf-env/bin/python


In [2]:
%pip install pydot
%pip install tensorflow
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle

import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from pathlib import Path

from constants import (
    DATA_INPUT_PATH,
    MODEL_PATH,
    METADATA_PATH,
)

CLASSES = {
    0: "resting",
    1: "palm up",
    2: "closed fist",
    3: "ok",
    4: "pointer finger",
    5: "peace",
    6: "shaa",
    7: "peace among worlds"
}

# Read the files in the data dir

In [4]:
# Read all of the files in the data folder
files_in_folder = Path(DATA_INPUT_PATH).glob("*.csv")

files = [x for x in files_in_folder]
print([file for file in files])

[PosixPath('data/emg_gestures_data_20250205_184712.csv'), PosixPath('data/emg_gestures_data_20250205_212859.csv'), PosixPath('data/emg_gestures_data_20250201_215436.csv'), PosixPath('data/emg_gestures_data_20250128_220435.csv'), PosixPath('data/emg_gestures_data_20250226_122030.csv'), PosixPath('data/emg_gestures_data_20250202_172502.csv'), PosixPath('data/emg_gestures_data_20250219_122322.csv')]


## Convert .csv(s) to dataframes and concatenate

In [32]:
import numpy as np
import pandas as pd

# List of file paths (update this list with your actual file paths)
files = [
    "data/emg_gestures_data_20250201_215436.csv",
    "data/emg_gestures_data_20250128_220435.csv",
    "data/emg_gestures_data_20250226_122030.csv",
    "data/emg_gestures_data_20250202_172502.csv",
    "data/emg_gestures_data_20250219_122322.csv"
]

# Parameters
window_size = 10  # Number of past samples to include
max_value = 255   # Target maximum value after scaling

# Expected columns
columns = ["gesture_id", "s1", "s2", "s3", "s4", "s5", "s6", "s7", "s8"]

# Container for normalized DataFrames
dfs = []

for file in files:
    # Read the data and select the desired columns
    df = pd.read_csv(str(file))
    df = df[columns]
    
    # Remove duplicates if needed
    df = df.drop_duplicates()
    
    # Print shape and first few rows before normalization
    print(f"\nFile: {file}")
    print("Shape before normalization:", df.shape)
    print("DataFrame before normalization (first 5 rows):")
    print(df.head())
    
    # Normalize the data before appending
    feature_columns = df.columns[1:]  # s1 to s8
    features = df[feature_columns].to_numpy()  # shape: (n_samples, 8)
    
    n_samples = features.shape[0]
    n_features = features.shape[1]
    reshaped_data = []
    
    # Build a 3D array: (samples, window_size, features)
    for i in range(n_samples):
        # Get the last `window_size` rows (or fewer) for each sample
        window = features[max(i - window_size + 1, 0): i + 1]
        # Pad with zeros if needed
        if window.shape[0] < window_size:
            padding = np.zeros((window_size - window.shape[0], n_features))
            window = np.vstack((padding, window))
        reshaped_data.append(window)
    
    # Convert list to numpy array and adjust dimensions to (samples, features, time)
    reshaped_data = np.stack(reshaped_data, axis=0)
    reshaped_data = reshaped_data.transpose(0, 2, 1)
    
    # Compute the RMS value for each sensor over the time axis for each sample
    rms_data = np.sqrt(np.mean(np.square(reshaped_data), axis=2))  # shape: (n_samples, features)
    
    # Compute global min and max for each sensor (over the current file's RMS data)
    min_val = np.min(rms_data, axis=0)
    max_val = np.max(rms_data, axis=0)
    
    # Prevent division by zero if any sensor has constant values
    range_val = np.where(max_val - min_val == 0, 1, max_val - min_val)
    
    # Apply old normalization method (per-sample RMS normalized using global min/max)
    normalized_data = ((rms_data - min_val) / range_val) * max_value
    normalized_data = normalized_data.round().astype(int)
    
    # Reconstruct the DataFrame with normalized sensor data
    df_transformed = df.copy()
    df_transformed[feature_columns] = normalized_data
    
    # Print shape and first few rows after normalization
    print("Shape after normalization:", df_transformed.shape)
    print("DataFrame after normalization (first 5 rows):")
    print(df_transformed.head())
    
    dfs.append(df_transformed)

# Concatenate all normalized DataFrames into one final DataFrame
df_final = pd.concat(dfs, axis=0)
print("\nFinal concatenated normalized DataFrame (first 5 rows):")
print(df_final.head())
print("Final shape:", df_final.shape)

df = df_final
print(len(df_final))


File: data/emg_gestures_data_20250201_215436.csv
Shape before normalization: (9803, 9)
DataFrame before normalization (first 5 rows):
   gesture_id   s1   s2   s3   s4   s5   s6   s7   s8
0           0  523  335  437  182  194  271  425  431
1           0  285  214  374  125  108  139  298  233
2           0  153  170  393  119   80  108  275  123
3           0  133  164  415  116   73   70  126   98
4           0  102  168  395  113   68   54   92   77
Shape after normalization: (9803, 9)
DataFrame after normalization (first 5 rows):
   gesture_id  s1  s2  s3  s4  s5  s6  s7  s8
0           0  69  27  14   6  21  22  41  69
1           0  80  34  23   9  27  26  52  80
2           0  83  38  31  12  30  28  61  83
3           0  85  41  38  15  32  29  62  84
4           0  87  44  44  17  34  29  63  86

File: data/emg_gestures_data_20250128_220435.csv
Shape before normalization: (19675, 9)
DataFrame before normalization (first 5 rows):
   gesture_id   s1   s2   s3   s4   s5   s6   

## Scale and clean the data, then train the model

In [31]:
X = df.drop(columns=['gesture_id'])
y = df['gesture_id']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Reshape the data for Conv1D
X_train_scaled = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_scaled = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

# # filter for valid classes because the data is not clean
# valid_classes = [0, 2, 3]
# mask_train = y_train.isin(valid_classes)
# mask_test = y_test.isin(valid_classes)

# # apply the mask to the training and testing data
# X_train_filtered = X_train_scaled[mask_train]
# y_train_filtered = y_train[mask_train]

# X_test_filtered = X_test_scaled[mask_test]
# y_test_filtered = y_test[mask_test]

# one hot encode the target data
y_train_categorical = to_categorical(y_train, num_classes=8)
y_test_categorical = to_categorical(y_test, num_classes=8)

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(8, activation='softmax')
])

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Compile the model with categorical crossentropy for multi-class classification
model.compile(
    optimizer=Adam(learning_rate=0.001), 
    loss='categorical_crossentropy', 
    metrics=['accuracy'],
)

# Train the model using the filtered training data
model.fit(
    X_train_scaled, 
    y_train_categorical, 
    epochs=100, 
    batch_size=64, 
    validation_split=0.2, 
    callbacks=[early_stopping]
)

# Evaluate the model on the filtered test set
test_loss, test_acc = model.evaluate(X_test_scaled, y_test_categorical)

print(f"Test accuracy: {test_acc}")

Epoch 1/100


/Users/ioana/.pyenv/versions/tf-env/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


590/590 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.6872 - loss: 0.9406 - val_accuracy: 0.9188 - val_loss: 0.2921
Epoch 2/100
590/590 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.8472 - loss: 0.4730 - val_accuracy: 0.9138 - val_loss: 0.3103
Epoch 3/100
590/590 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.8211 - loss: 0.5570 - val_accuracy: 0.8694 - val_loss: 0.4345
Epoch 4/100
590/590 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.7704 - loss: 0.7636 - val_accuracy: 0.8303 - val_loss: 0.5384
Epoch 5/100
590/590 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.7113 - loss: 1.0753 - val_accuracy: 0.8293 - val_loss: 0.7430
Epoch 6/100
590/590 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.6683 - loss: 1.2792 - val_accuracy: 0.7833 - val_loss: 0.6826
369/369 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9162 - loss: 0.2956
Test accuracy: 0.9206308722496033


In [ ]:
# Save the model
model.save(f"model/{MODEL_PATH}")

# Save the scaler and the column names to a pickle file
with open(f"model/{METADATA_PATH}", 'wb') as f:
    pickle.dump((scaler, X_train.columns), f)

In [ ]:
# thumbs-up then peace-sign
emg_data = [
    [160,245,126,32,28,25,26,99], # thumbs-up
    [67,197,559,104,41,82,257,169], # peace-sign
    [205,440,165,40,27,83,229,226], # gun-fingers
    [416,434,134,71,36,48,102,103] # fist
]

for data in emg_data:
    emg_features_df = pd.DataFrame([data], columns=X_train.columns)

    emg_features_scaled = scaler.transform(emg_features_df)
    emg_features_reshaped = emg_features_scaled.reshape(1, -1)

    prediction = model.predict(emg_features_reshaped)
    predicted_class = np.argmax(prediction)

    print(f"Predicted class: {predicted_class} - {CLASSES[predicted_class]}")
